**PREPARING THE FEATURES**

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.read_csv("train.csv")

#Missing means "the house has no such feature": fill with "None"
no_feature_cols = ["PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
                   "GarageType", "GarageFinish", "GarageQual", "GarageCond",
                   "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
                   "MasVnrType"]
df[no_feature_cols] = df[no_feature_cols].fillna("None")

# Numeric columns where missing means "none"
df["MasVnrArea"] = df["MasVnrArea"].fillna(0)
df["GarageYrBlt"] = df["GarageYrBlt"].fillna(0)

# One missing value: use the most common value
df["Electrical"] = df["Electrical"].fillna(df["Electrical"].mode()[0])

# Same split as before, so results stay comparable
train, test = train_test_split(df, test_size=0.2, random_state=42)
train = train.copy()
test = test.copy()

# LotFrontage: fill with the median of the TRAINING houses only
median_frontage = train["LotFrontage"].median()
train["LotFrontage"] = train["LotFrontage"].fillna(median_frontage)
test["LotFrontage"] = test["LotFrontage"].fillna(median_frontage)

print("Missing values left in training set:", train.isna().sum().sum())
print("Missing values left in test set:", test.isna().sum().sum())
print("Training houses:", len(train), "| Test houses:", len(test))


Missing values left in training set: 0
Missing values left in test set: 0
Training houses: 1168 | Test houses: 292


*The test houses stand in for houses the model has never seen. If the median came from all 1,460 houses, a little information from the test set would leak into the training, and the score would look slightly better than it really is.*

**ENCODE THE TEXT COLUMN AND REFIT THE MODEL**

In [2]:
y_train = train["SalePrice"]
y_test = test["SalePrice"]

base_cols = ["GrLivArea", "OverallQual", "GarageCars", "YearBuilt", "TotalBsmtSF"]

def score(Xtr, Xte, name):
  m = LinearRegression().fit(Xtr, y_train)
  p = m.predict(Xte)
  print(name, "| MAE:", round(mean_absolute_error(y_test, p)), "| R2:", round(r2_score(y_test, p), 2))


# 1. Five numeric features (your Day 16 result, for reference)
score(train[base_cols], test[base_cols], "Five numeric features")

# 2. Add neighboorhood
hood_train = pd.get_dummies(train["Neighborhood"], prefix = "Hood", dtype=int)
hood_test = pd.get_dummies(test["Neighborhood"], prefix="Hood", dtype=int)
hood_test = hood_test.reindex(columns=hood_train.columns, fill_value=0)
score(pd.concat([train[base_cols], hood_train], axis=1),
      pd.concat([test[base_cols], hood_test], axis=1),
      "+ Neighborhood")

# 3. Every column
X_train_all = pd.get_dummies(train.drop(columns=["SalePrice", "Id"]), dtype=int)
X_test_all = pd.get_dummies(test.drop(columns=["SalePrice", "Id"]), dtype=int)
X_test_all = X_test_all.reindex(columns=X_train_all.columns, fill_value=0)
print("Columns after encoding:", X_train_all.shape[1])
score(X_train_all, X_test_all, "All columns")

Five numeric features | MAE: 25415 | R2: 0.79
+ Neighborhood | MAE: 22339 | R2: 0.83
Columns after encoding: 300
All columns | MAE: 21108 | R2: 0.44


In [3]:
m_all = LinearRegression().fit(X_train_all, y_train)
p_all = m_all.predict(X_test_all)

check = pd.DataFrame({
    "actual": y_test.values,
    "predicted": p_all.round(),
    "Neighborhood": test["Neighborhood"].values,
    "GrLivArea": test["GrLivArea"].values,
})
check["error"] = (check["predicted"] - check["actual"]).abs()

print("Median error:", round(check["error"].median()))
check.sort_values("error", ascending=False).head(5)

Median error: 12466


,actual,predicted,Neighborhood,GrLivArea,error
193,171000,-840462.0,Mitchel,1118,1011462.0
139,755000,568368.0,NoRidge,4316,186632.0
74,611657,468318.0,NridgHt,2364,143339.0
168,556581,420565.0,StoneBr,2868,136016.0
43,253293,372565.0,NridgHt,2042,119272.0


In [8]:
contrib = pd.Series(m_all.coef_ * X_test_all.iloc[193].values, index=X_test_all.columns)
contrib[contrib.abs().sort_values(ascending=False).head(5).index].round()

,0
PoolArea,3660301.0
PoolQC_Gd,-1632299.0
YrSold,-1001323.0
YearBuilt,625477.0
YearRemodAdd,301562.0


In [9]:
print("Training houses with a pool:", (train["PoolArea"] > 0).sum())
print("Test house 193 pool area:", test.iloc[193]["PoolArea"], "| pool quantity:", test.iloc[193]["PoolQC"])

Training houses with a pool: 6
Test house 193 pool area: 576 | pool quantity: Gd


In [10]:
keep = [c for c in X_train_all.columns if c != "PoolArea" and not c.startswith("PoolQC")]
score(X_train_all[keep], X_test_all[keep], "All columns without pool")

All columns without pool | MAE: 19245 | R2: 0.88


*The Mitchel house’s prediction of negative 840,462 came from the pool columns: very few houses have pools, so the model gave them extreme weights (PoolArea +3,660,301 and PoolQC_Gd -1,632,299) that failed to cancel for this house. Dropping the pool columns [fixes / improves] the model. Rare features need care before they go into a linear model.*

*Dropping the pool columns fixed the problem: with only 6 training houses having a pool, the model had given PoolArea and PoolQC extreme weights. Without them, the all-columns model has an average error of 19,245 (R² 0.88), about 69% lower than the baseline. Caveat: I chose to drop these columns after seeing which test house failed, so the test score is slightly optimistic. Rare features need care before they go into a linear model.*